# 10 · Combining Data — Merge, Join, Concat
*Intro to Python for Scientists & Public Health Professionals*

Real analyses usually pull from more than one table. Three tools cover almost everything:

- **`concat`** — stack tables (rows on top of each other, or columns side by side)
- **`merge`** — SQL-style join on one or more **key columns**
- **`join`** — merge on the **index**

### By the end of this notebook you can
- Stack tables with `concat`
- Merge on shared keys, and choose inner / left / right / outer
- Merge when the key columns have different names
- Combine two real county-level public-health datasets

### Agenda
1. Concatenate
2. Merge basics
3. Join types (inner / left / right / outer)
4. Keys with different names
5. A real merge: ACS demographics × CDC PLACES health
6. `join` on the index

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

In [ ]:
import numpy as np
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"

## 1. Concatenate

`concat` stacks like-shaped tables. Pass a **list** of DataFrames. (The old `df.append()` method was removed in pandas 2.0 — use `concat`.)

In [ ]:
oct_df = pd.DataFrame({"site": ["Riverside", "Hilltop"], "month": "Oct", "visits": [4200, 2600]})
nov_df = pd.DataFrame({"site": ["Riverside", "Hilltop"], "month": "Nov", "visits": [3900, 2500]})

pd.concat([oct_df, nov_df], ignore_index=True)   # stack rows; renumber the index

> Reference: [Merge, join, concatenate](https://pandas.pydata.org/docs/user_guide/merging.html).

## 2. Merge basics

`merge` joins on a shared **key column**. By default it's an **inner** join — only keys present in *both* tables survive.

In [ ]:
demographics = pd.DataFrame({"county": ["Appling", "Bacon", "Bibb"],
                             "population": [18500, 11000, 157000]})
health = pd.DataFrame({"county": ["Appling", "Bacon", "Cobb"],
                       "diabetes_pct": [16.3, 15.1, 9.8]})

demographics.merge(health, on="county", how="inner")   # only Appling & Bacon are in both

## 3. Join types

`how=` controls which keys are kept. Unmatched rows get `NaN` for the missing side.

- **inner** — keys in both (default)
- **left** — all left rows
- **right** — all right rows
- **outer** — keys in either

In [ ]:
demographics.merge(health, on="county", how="left")    # keeps Bibb; its diabetes_pct is NaN

In [ ]:
demographics.merge(health, on="county", how="outer")   # keeps Bibb AND Cobb; NaNs fill the gaps

## 4. Keys with different names

When the key is spelled differently in each table, use `left_on` / `right_on`.

In [ ]:
a = pd.DataFrame({"county": ["Appling", "Bacon"], "pop": [18500, 11000]})
b = pd.DataFrame({"cty": ["Appling", "Bacon"], "rate": [16.3, 15.1]})

a.merge(b, left_on="county", right_on="cty")

## 5. A real merge: demographics × health

Two county-level datasets: **ACS** (Census demographics) and **CDC PLACES** (health estimates). Each county has a FIPS code — the natural join key (`CountyId` in ACS, `CountyFIPS` in PLACES).

In [ ]:
acs    = pd.read_csv(f"{BASE_URL}/acs2017.csv")   # demographics
places = pd.read_csv(f"{BASE_URL}/places.csv")    # health estimates
print("acs:", acs.shape, "| places:", places.shape)

In [ ]:
# inner join: counties present in BOTH sources
both = acs.merge(places, left_on="CountyId", right_on="CountyFIPS", how="inner")
print("matched counties:", both.shape[0])

# outer join: every county from EITHER source (non-matches get NaN)
allc = acs.merge(places, left_on="CountyId", right_on="CountyFIPS", how="outer")
print("union of counties:", allc.shape[0])

> Key-matching is the #1 merge gotcha: the join keys must be the **same dtype**. Here both FIPS columns are integers, so they compare correctly. If one were text (`"13001"`) and the other an integer (`13001`), the merge would silently match nothing — always check.

### Exercise 1 — Merge one state, two ways *(12 min)*

Filter ACS and PLACES to **Georgia** (ACS uses `State == "Georgia"`; PLACES uses `StateAbbr == "GA"`). Both have 159 counties. Merge them on the FIPS key with an **inner** join, then an **outer** join. Do the row counts differ? Do both sources cover the same counties?

In [ ]:
# Your work here


## 6. `join` on the index

`merge` handles most cases. `join` is the shortcut when you're combining on the **index** rather than a column.

In [ ]:
left  = demographics.set_index("county")
right = health.set_index("county")

left.join(right, how="inner")   # aligns on the shared index

### Capstone Part 3

Open your **Rural Hospital Closures capstone** and complete **Part 3 (Merge)**: read `payment_types.csv` and join it onto your cleaned data so each closure gains a readable `description`.

## Wrap-up

You can stack tables with `concat`, merge on shared (or differently-named) keys, choose the right join type, and combine real datasets on a common key — while watching for dtype mismatches.

**Next:** Reshaping — melt, pivot, and stack to change a table's layout.